In [71]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_test.head()

,customer_id,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio
0,CUST146767,39,Self-Employed,1.7,36113,595,4.4,11,4273,0,2,1,Credit Card,Personal,44600,21.32,0.118,1.235,0.412
1,CUST136829,38,Employed,9.4,44116,536,12.6,1009,7060,0,2,1,Personal Loan,Home Improvement,38800,15.63,0.160,0.879,0.293
2,CUST119409,18,Self-Employed,1.0,76459,745,0.4,72,6221,0,0,0,Personal Loan,Personal,19100,10.89,0.081,0.250,0.083
3,CUST112393,25,Self-Employed,2.0,86494,569,5.1,414,57211,1,1,0,Credit Card,Medical,21300,22.51,0.661,0.246,0.082
4,CUST143342,34,Student,0.8,29647,608,0.2,21,7391,0,0,0,Line of Credit,Medical,14500,11.91,0.249,0.489,0.163


In [72]:
#subtasks
subtask1 = []
subtask2 = []

def f1(val):
    if val < 30:
        return "Young"
    elif val < 60:
        return "Adult"
    else:
        return "Senior"

def f2(val):
    if val < 20:
        return "LowRisk"
    elif val < 40:
        return "MediumRisk"
    else:
        return "HighRisk"

subtask1 = df_test.age.apply(f1)
subtask2 = df_test.debt_to_income_ratio.apply(f2)

subtask3 = df_test.current_debt + df_test.derogatory_marks + df_test.delinquencies_last_2yrs

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.customer_id,
    "answer": subtask1
})

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.customer_id,
    "answer": subtask2
})

df_3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.customer_id,
    "answer": subtask3
})

subtasks = pd.concat([df_1, df_2, df_3])
subtasks

,subtaskID,datapointID,answer
0,1,CUST146767,Adult
1,1,CUST136829,Adult
2,1,CUST119409,Young
3,1,CUST112393,Young
4,1,CUST143342,Adult
...,...,...,...
9995,3,CUST116850,11854
9996,3,CUST128287,4014
9997,3,CUST121440,1678
9998,3,CUST127749,19471


In [73]:
df_train.columns

Index(['customer_id', 'age', 'occupation_status', 'years_employed',
       'annual_income', 'credit_score', 'credit_history_years',
       'savings_assets', 'current_debt', 'defaults_on_file',
       'delinquencies_last_2yrs', 'derogatory_marks', 'product_type',
       'loan_intent', 'loan_amount', 'interest_rate', 'debt_to_income_ratio',
       'loan_to_income_ratio', 'payment_to_income_ratio', 'loan_status'],
      dtype='object')

In [74]:
#preprocessing

df_test['obligations'] = df_test.current_debt + df_test.derogatory_marks + df_test.delinquencies_last_2yrs
df_train['obligations'] = df_train.current_debt + df_train.derogatory_marks + df_train.delinquencies_last_2yrs

from sklearn.preprocessing import LabelEncoder
to_encode = ["occupation_status", "product_type", "loan_intent"]
for col in to_encode:
    enc = LabelEncoder()
    df_train[col] = enc.fit_transform(df_train[col])
    df_test[col] = enc.transform(df_test[col])

features = ['age', 'occupation_status', 'years_employed', 'annual_income', 'credit_score', 'credit_history_years', 'savings_assets', 'current_debt', 'defaults_on_file', 'obligations',
       'delinquencies_last_2yrs', 'derogatory_marks', 'product_type', 'loan_intent', 'loan_amount', 'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio', 'payment_to_income_ratio']
target = 'loan_status'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

X.head()

,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,obligations,delinquencies_last_2yrs,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio
0,40,1,14.1,77335,679,4.4,67,17026,0,17026,0,0,2,2,26200,12.04,0.220,0.339,0.113
1,18,0,0.0,34946,639,1.2,292,2620,0,2621,1,0,2,2,37400,14.95,0.075,1.070,0.357
2,29,0,9.4,26778,595,5.5,191,4702,1,4703,1,0,2,5,52300,15.51,0.176,1.953,0.651
3,27,2,1.5,25624,590,5.7,414,2609,0,2609,0,0,2,1,4800,15.91,0.102,0.187,0.062
4,18,0,0.0,25676,568,1.8,19,3453,1,3455,1,1,2,4,24700,15.47,0.134,0.962,0.321


In [81]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

model = CatBoostClassifier(random_state=42, silent=True)

cross_val_score(model, X, y, cv=5, scoring='roc_auc')

array([0.98236176, 0.98332199, 0.98087876, 0.98217886, 0.98527219])

In [86]:
model.fit(X, y)
y_final = model.predict_proba(X_final)[:, 1]

In [87]:
df_preds = pd.DataFrame({
    "subtaskID": 4,
    "datapointID": df_test.customer_id,
    "answer": y_final
})

df_preds.head()

,subtaskID,datapointID,answer
0,4,CUST146767,1.191893e-03
1,4,CUST136829,5.386435e-03
2,4,CUST119409,9.915887e-01
3,4,CUST112393,6.087589e-07
4,4,CUST143342,3.290529e-01


In [88]:
submission = pd.concat([subtasks, df_preds])
submission

,subtaskID,datapointID,answer
0,1,CUST146767,Adult
1,1,CUST136829,Adult
2,1,CUST119409,Young
3,1,CUST112393,Young
4,1,CUST143342,Adult
...,...,...,...
9995,4,CUST116850,0.988553
9996,4,CUST128287,0.0
9997,4,CUST121440,0.372756
9998,4,CUST127749,0.999778


In [89]:
submission.to_csv("submission.csv", index=False)